In [0]:
landing_table=dbutils.widgets.get("landing_table")
landing_flattened_table=dbutils.widgets.get("landing_flattened_table")
raw_table=dbutils.widgets.get("raw_table")
volume_path=dbutils.widgets.get("volume_path")

In [0]:
try:
    spark.sql(f"""
        WITH new_records AS (
        SELECT src.*
        FROM {landing_table} src
        LEFT OUTER JOIN {landing_flattened_table} tgt
        ON src.transID = tgt.transID
        AND src._file_name = tgt._file_name
        WHERE tgt.transID IS NULL
        ),
        new_records_dedup AS (
        SELECT
            AdjAmounts_expl AS AdjAmounts,
            Adjustments_expl.AdjAmount AS Adjustments_AdjAmount,
            Adjustments_expl.AdjQuantity AS Adjustments_AdjQuantity,
            Adjustments_expl.GroupCode AS Adjustments_GroupCode,
            Adjustments_expl.ReasonCode AS Adjustments_ReasonCode,
            AppealCreated,
            AppealCreation,
            AppealResults_expl AS AppealResults,
            AppealType,
            AppealTypeName,
            AppealTypes_expl AS AppealTypes,
            AssignedToID,
            Balanced,
            ClaimCharge,
            ClaimFilingCode,
            ClaimFreqCode,
            ClaimPayment,
            ClassContractNo,
            CoverageAmt,
            CrcPatientFirst,
            CrcPatientID.ChangedID AS CrcPatientID_ChangedID,
            CrcPatientLast,
            CrcPayerID.PayerID AS CrcPayerID_PayerID,
            CrcPayerName,
            CreateDate,
            CreateMode,
            CrossoverCarrier,
            CrossoverCarrierID.PayerID AS CrossoverCarrierID_PayerID,
            CustomGroup,
            DRGCode,
            DRGWeight,
            DischargeFraction,
            EditByID,
            EditDate,
            ExpirationDate,
            FacilityType,
            GroupNo,
            ImportBatchID,
            Interest,
            LastUpdated,
            LastUpdatedUserId,
            LastUserUpdatedDate,
            MedRecordNo,
            MediaCode,
            MemberID,
            OrigRefNo,
            ParentID,
            PatID,
            PatientCtlNo,
            PatientFirst,
            PatientID.MedicaidID AS PatientID_MedicaidID,
            PatientID.MemberID AS PatientID_MemberID,
            PatientLast,
            PatientMiddle,
            PatientPaid,
            PatientResp,
            PayerCtlNo,
            PayerMatchID,
            PayerPartnerID,
            PriorAuthNo,
            ProvMatchID,
            ProvPartnerID,
            ReceivedDate,
            ReimbursementAmount,
            RemarkCodes_expl.RemarkCode AS RemarkCodes_RemarkCode,
            RemarkCodes_expl.RemarkType AS RemarkCodes_RemarkType,
            RemitClaimID,
            RemitGroupID,
            RemitPaymentID,
            RenderingProvFirst,
            RenderingProvID.NPI AS RenderingProvID_NPI,
            RenderingProvID.ProvCommNo AS RenderingProvID_ProvCommNo,
            RenderingProvLast,
            Services_expl.AdjHCPC AS Services_AdjHCPC,
            Services_Adjustments_expl.AdjAmount AS Services_Adjustments_AdjAmount,
            Services_Adjustments_expl.AdjQuantity AS Services_Adjustments_AdjQuantity,
            Services_Adjustments_expl.GroupCode AS Services_Adjustments_GroupCode,
            Services_Adjustments_expl.ReasonCode AS Services_Adjustments_ReasonCode,
            Services_expl.Allowed AS Services_Allowed,
            Services_expl.AuthNo AS Services_AuthNo,
            Services_expl.Balanced AS Services_Balanced,
            Services_expl.Charge AS Services_Charge,
            Services_expl.HCPC AS Services_HCPC,
            Services_expl.LineNo AS Services_LineNo,
            Services_expl.LocationNo AS Services_LocationNo,
            Services_expl.Modifier1 AS Services_Modifier1,
            Services_expl.Modifier2 AS Services_Modifier2,
            Services_expl.Modifier3 AS Services_Modifier3,
            Services_expl.Modifier4 AS Services_Modifier4,
            Services_expl.Payment AS Services_Payment,
            Services_expl.ProvCtlNo AS Services_ProvCtlNo,
            Services_expl.ReimbursementAmount AS Services_ReimbursementAmount,
            Services_RemarkCodes_expl.RemarkCode AS Services_RemarkCodes_RemarkCode,
            Services_RemarkCodes_expl.RemarkType AS Services_RemarkCodes_RemarkType,
            Services_expl.RenderingProvID.CommercialNo AS Services_RenderingProvID_CommercialNo,
            Services_expl.RenderingProvID.NPI AS Services_RenderingProvID_NPI,
            Services_expl.RevenueCode AS Services_RevenueCode,
            Services_expl.ServiceEnd AS Services_ServiceEnd,
            Services_expl.ServiceStart AS Services_ServiceStart,
            Services_expl.Underpayment AS Services_Underpayment,
            Services_expl.UnderpaymentAmount AS Services_UnderpaymentAmount,
            Services_expl.Units AS Services_Units,
            Services_expl.UnitsPaid AS Services_UnitsPaid,
            SourceFileID,
            StatementEnd,
            StatementStart,
            StatusCode,
            SubscriberFirst,
            SubscriberID.MemberID AS SubscriberID_MemberID,
            SubscriberLast,
            SubscriberMiddle,
            TransID,
            TransStatus,
            TransType,
            TransmitDate,
            Underpayment,
            UnderpaymentAmount,
            WorkQueue,
            _file_name,
            _load_timestamp
        FROM new_records
        LATERAL VIEW explode_outer(AdjAmounts) AdjAmounts_lv AS AdjAmounts_expl
        LATERAL VIEW explode_outer(Adjustments) Adjustments_lv AS Adjustments_expl
        LATERAL VIEW explode_outer(AppealResults) AppealResults_lv AS AppealResults_expl
        LATERAL VIEW explode_outer(AppealTypes) AppealTypes_lv AS AppealTypes_expl
        LATERAL VIEW explode_outer(RemarkCodes) RemarkCodes_lv AS RemarkCodes_expl
        LATERAL VIEW explode_outer(Services) Services_lv AS Services_expl
        LATERAL VIEW explode_outer(Services_expl.Adjustments) Services_Adjustments_lv AS Services_Adjustments_expl
        LATERAL VIEW explode_outer(Services_expl.RemarkCodes) Services_RemarkCodes_lv AS Services_RemarkCodes_expl
        ),
        new_records_final AS (
        SELECT
            COALESCE(TRY_CAST(AdjAmounts AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS AdjAmounts,
            COALESCE(TRY_CAST(Adjustments_AdjAmount AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Adjustments_AdjAmount,
            COALESCE(TRY_CAST(Adjustments_AdjQuantity AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Adjustments_AdjQuantity,
            COALESCE(TRY_CAST(Adjustments_GroupCode AS STRING),'') AS Adjustments_GroupCode,
            COALESCE(TRY_CAST(Adjustments_ReasonCode AS STRING),'') AS Adjustments_ReasonCode,
            COALESCE(TRY_CAST(AppealCreated AS STRING),'') AS AppealCreated,
            COALESCE(TRY_CAST(AppealCreation AS STRING),'') AS AppealCreation,
            COALESCE(TRY_CAST(AppealResults AS STRING),'') AS AppealResults,
            COALESCE(TRY_CAST(AppealType AS STRING),'') AS AppealType,
            COALESCE(TRY_CAST(AppealTypeName AS STRING),'') AS AppealTypeName,
            COALESCE(TRY_CAST(AppealTypes AS STRING),'') AS AppealTypes,
            COALESCE(TRY_CAST(AssignedToID AS BIGINT),0) AS AssignedToID,
            COALESCE(TRY_CAST(Balanced AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Balanced,
            COALESCE(TRY_CAST(ClaimCharge AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS ClaimCharge,
            COALESCE(TRY_CAST(ClaimFilingCode AS STRING),'') AS ClaimFilingCode,
            COALESCE(TRY_CAST(ClaimFreqCode AS STRING),'') AS ClaimFreqCode,
            COALESCE(TRY_CAST(ClaimPayment AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS ClaimPayment,
            COALESCE(TRY_CAST(ClassContractNo AS STRING),'') AS ClassContractNo,
            COALESCE(TRY_CAST(CoverageAmt AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS CoverageAmt,
            COALESCE(TRY_CAST(CrcPatientFirst AS STRING),'') AS CrcPatientFirst,
            COALESCE(TRY_CAST(CrcPatientID_ChangedID AS BIGINT),0) AS CrcPatientID_ChangedID,
            COALESCE(TRY_CAST(CrcPatientLast AS STRING),'') AS CrcPatientLast,
            COALESCE(TRY_CAST(CrcPayerID_PayerID AS BIGINT),0) AS CrcPayerID_PayerID,
            COALESCE(TRY_CAST(CrcPayerName AS STRING),'') AS CrcPayerName,
            COALESCE(TRY_CAST(CreateDate AS STRING),'') AS CreateDate,
            COALESCE(TRY_CAST(CreateMode AS STRING),'') AS CreateMode,
            COALESCE(TRY_CAST(CrossoverCarrier AS STRING),'') AS CrossoverCarrier,
            COALESCE(TRY_CAST(CrossoverCarrierID_PayerID AS BIGINT),0) AS CrossoverCarrierID_PayerID,
            COALESCE(TRY_CAST(CustomGroup AS STRING),'') AS CustomGroup,
            COALESCE(TRY_CAST(DRGCode AS STRING),'') AS DRGCode,
            COALESCE(TRY_CAST(DRGWeight AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS DRGWeight,
            COALESCE(TRY_CAST(DischargeFraction AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS DischargeFraction,
            COALESCE(TRY_CAST(EditByID AS BIGINT),0) AS EditByID,
            COALESCE(TRY_CAST(EditDate AS STRING),'') AS EditDate,
            COALESCE(TRY_CAST(ExpirationDate AS STRING),'') AS ExpirationDate,
            COALESCE(TRY_CAST(FacilityType AS STRING),'') AS FacilityType,
            COALESCE(TRY_CAST(GroupNo AS STRING),'') AS GroupNo,
            COALESCE(TRY_CAST(ImportBatchID AS BIGINT),0) AS ImportBatchID,
            COALESCE(TRY_CAST(Interest AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Interest,
            COALESCE(TRY_CAST(LastUpdated AS STRING),'') AS LastUpdated,
            COALESCE(TRY_CAST(LastUpdatedUserId AS BIGINT),0) AS LastUpdatedUserId,
            COALESCE(TRY_CAST(LastUserUpdatedDate AS STRING),'') AS LastUserUpdatedDate,
            COALESCE(TRY_CAST(MedRecordNo AS STRING),'') AS MedRecordNo,
            COALESCE(TRY_CAST(MediaCode AS STRING),'') AS MediaCode,
            COALESCE(TRY_CAST(MemberID AS BIGINT),0) AS MemberID,
            COALESCE(TRY_CAST(OrigRefNo AS STRING),'') AS OrigRefNo,
            COALESCE(TRY_CAST(ParentID AS BIGINT),0) AS ParentID,
            COALESCE(TRY_CAST(PatID AS BIGINT),0) AS PatID,
            COALESCE(TRY_CAST(PatientCtlNo AS STRING),'') AS PatientCtlNo,
            COALESCE(TRY_CAST(PatientFirst AS STRING),'') AS PatientFirst,
            COALESCE(TRY_CAST(PatientID_MedicaidID AS BIGINT),0) AS PatientID_MedicaidID,
            COALESCE(TRY_CAST(PatientID_MemberID AS STRING),'') AS PatientID_MemberID,
            COALESCE(TRY_CAST(PatientLast AS STRING),'') AS PatientLast,
            COALESCE(TRY_CAST(PatientMiddle AS STRING),'') AS PatientMiddle,
            COALESCE(TRY_CAST(PatientResp AS STRING),'') AS PatientResp,
            COALESCE(TRY_CAST(PayerCtlNo AS STRING),'') AS PayerCtlNo,
            COALESCE(TRY_CAST(PayerMatchID AS BIGINT),0) AS PayerMatchID,
            COALESCE(TRY_CAST(PayerPartnerID AS BIGINT),0) AS PayerPartnerID,
            COALESCE(TRY_CAST(PriorAuthNo AS STRING),'') AS PriorAuthNo,
            COALESCE(TRY_CAST(ProvMatchID AS BIGINT),0) AS ProvMatchID,
            COALESCE(TRY_CAST(ProvPartnerID AS BIGINT),0) AS ProvPartnerID,
            COALESCE(TRY_CAST(ReceivedDate AS STRING),'') AS ReceivedDate,
            COALESCE(TRY_CAST(RemarkCodes_RemarkCode AS STRING),'') AS RemarkCodes_RemarkCode,
            COALESCE(TRY_CAST(RemarkCodes_RemarkType AS STRING),'') AS RemarkCodes_RemarkType,
            COALESCE(TRY_CAST(RemitClaimID AS BIGINT),0) AS RemitClaimID,
            COALESCE(TRY_CAST(RemitGroupID AS BIGINT),0) AS RemitGroupID,
            COALESCE(TRY_CAST(RemitPaymentID AS BIGINT),0) AS RemitPaymentID,
            COALESCE(TRY_CAST(RenderingProvFirst AS STRING),'') AS RenderingProvFirst,
            COALESCE(TRY_CAST(RenderingProvID_NPI AS BIGINT),0) AS RenderingProvID_NPI,
            COALESCE(TRY_CAST(RenderingProvID_ProvCommNo AS BIGINT),0) AS RenderingProvID_ProvCommNo,
            COALESCE(TRY_CAST(RenderingProvLast AS STRING),'') AS RenderingProvLast,
            COALESCE(TRY_CAST(Services_AdjHCPC AS STRING),'') AS Services_AdjHCPC,
            COALESCE(TRY_CAST(Services_Adjustments_AdjAmount AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Adjustments_AdjAmount,
            COALESCE(TRY_CAST(Services_Adjustments_AdjQuantity AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Adjustments_AdjQuantity,
            COALESCE(TRY_CAST(Services_Adjustments_GroupCode AS STRING),'') AS Services_Adjustments_GroupCode,
            COALESCE(TRY_CAST(Services_Adjustments_ReasonCode AS STRING),'') AS Services_Adjustments_ReasonCode,
            COALESCE(TRY_CAST(Services_Allowed AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Allowed,
            COALESCE(TRY_CAST(Services_AuthNo AS STRING),'') AS Services_AuthNo,
            COALESCE(TRY_CAST(Services_Balanced AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Balanced,
            COALESCE(TRY_CAST(Services_Charge AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Charge,
            COALESCE(TRY_CAST(Services_HCPC AS STRING),'') AS Services_HCPC,
            COALESCE(TRY_CAST(Services_LineNo AS BIGINT),0) AS Services_LineNo,
            COALESCE(TRY_CAST(Services_LocationNo AS STRING),'') AS Services_LocationNo,
            COALESCE(TRY_CAST(Services_Modifier1 AS STRING),'') AS Services_Modifier1,
            COALESCE(TRY_CAST(Services_Modifier2 AS STRING),'') AS Services_Modifier2,
            COALESCE(TRY_CAST(Services_Modifier3 AS STRING),'') AS Services_Modifier3,
            COALESCE(TRY_CAST(Services_Modifier4 AS STRING),'') AS Services_Modifier4,
            COALESCE(TRY_CAST(Services_Payment AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Payment,
            COALESCE(TRY_CAST(Services_ProvCtlNo AS STRING),'') AS Services_ProvCtlNo,
            COALESCE(TRY_CAST(Services_RemarkCodes_RemarkCode AS STRING),'') AS Services_RemarkCodes_RemarkCode,
            COALESCE(TRY_CAST(Services_RemarkCodes_RemarkType AS STRING),'') AS Services_RemarkCodes_RemarkType,
            COALESCE(TRY_CAST(Services_RenderingProvID_CommercialNo AS BIGINT),0) AS Services_RenderingProvID_CommercialNo,
            COALESCE(TRY_CAST(Services_RenderingProvID_NPI AS BIGINT),0) AS Services_RenderingProvID_NPI,
            COALESCE(TRY_CAST(Services_RevenueCode AS STRING),'') AS Services_RevenueCode,
            COALESCE(TRY_CAST(Services_ServiceEnd AS STRING),'') AS Services_ServiceEnd,
            COALESCE(TRY_CAST(Services_ServiceStart AS STRING),'') AS Services_ServiceStart,
            COALESCE(TRY_CAST(Services_Underpayment AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Underpayment,
            COALESCE(TRY_CAST(Services_UnderpaymentAmount AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_UnderpaymentAmount,
            COALESCE(TRY_CAST(Services_Units AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_Units,
            COALESCE(TRY_CAST(Services_UnitsPaid AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Services_UnitsPaid,
            COALESCE(TRY_CAST(SourceFileID AS STRING),'') AS SourceFileID,
            COALESCE(TRY_CAST(StatementEnd AS STRING),'') AS StatementEnd,
            COALESCE(TRY_CAST(StatementStart AS STRING),'') AS StatementStart,
            COALESCE(TRY_CAST(StatusCode AS STRING),'') AS StatusCode,
            COALESCE(TRY_CAST(SubscriberFirst AS STRING),'') AS SubscriberFirst,
            COALESCE(TRY_CAST(SubscriberID_MemberID AS BIGINT),0) AS SubscriberID_MemberID,
            COALESCE(TRY_CAST(SubscriberLast AS STRING),'') AS SubscriberLast,
            COALESCE(TRY_CAST(SubscriberMiddle AS STRING),'') AS SubscriberMiddle,
            COALESCE(TRY_CAST(TransID AS BIGINT),0) AS TransID,
            COALESCE(TRY_CAST(TransStatus AS STRING),'') AS TransStatus,
            COALESCE(TRY_CAST(TransType AS STRING),'') AS TransType,
            COALESCE(TRY_CAST(TransmitDate AS STRING),'') AS TransmitDate,
            COALESCE(TRY_CAST(Underpayment AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS Underpayment,
            COALESCE(TRY_CAST(UnderpaymentAmount AS DECIMAL(10,2)),CAST(0 AS DECIMAL(10,2))) AS UnderpaymentAmount,
            COALESCE(TRY_CAST(WorkQueue AS STRING),'') AS WorkQueue,
            COALESCE(TRY_CAST(_file_name AS STRING),'') AS _file_name,
            COALESCE(TRY_CAST(_load_timestamp AS TIMESTAMP),current_timestamp()) AS _load_timestamp
        FROM new_records_dedup
        )

        INSERT INTO {landing_flattened_table}
        SELECT * FROM new_records_final
    """)
except Exception as e:
    print(f"Error loading into {landing_flattened_table}: {e}")
    raise 